In [ ]:
%cd /home/dhuruva/projects/ctb-emuller/dhuruva/plastyfire/new_fitting

: 

In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np

# Load data
with open("/home/dhuruva/projects/ctb-emuller/dhuruva/plastyfire/trace_results/Chindemi_params/180164-197248/10Hz_5ms/simulation_traces.pkl", "rb") as f:
    data = pickle.load(f)

# Get time and effcai
t = data.get("t", np.arange(data["effcai_GB"].shape[-1]))
effcai = np.asarray(data["effcai_GB"])

# Ensure shape is (n_synapses, n_timepoints)
if effcai.shape[0] == len(t) and effcai.shape[1] != len(t):
    effcai = effcai.T

# Plot
plt.figure(figsize=(12, 4))
for i in range(min(10, effcai.shape[0])):  # plot up to 10 synapses
    plt.plot(t / 1000, effcai[i], alpha=0.7)
    break
plt.xlabel("Time (s)")
plt.ylabel("effCa_i (μM)")
plt.title("Effective Intracellular Calcium")
plt.tight_layout()
plt.axis([0,50,0,0.22])

In [ ]:
# Get time and cai
t = data.get("t", np.arange(data["cai_CR"].shape[-1]))
cai = np.asarray(data["cai_CR"])

# Ensure shape is (n_synapses, n_timepoints)
if cai.shape[0] == len(t) and cai.shape[1] != len(t):
    cai = cai.T

# Plot
plt.figure(figsize=(12, 4))
for i in range(min(10, cai.shape[0])):  # plot up to 10 synapses
    plt.plot(t / 1000, cai[i], alpha=0.7)
    break
plt.xlabel("Time (s)")
plt.ylabel("Ca_i (μM)")
plt.title("Calcium Concentration")
plt.tight_layout()
plt.axis([0, 50, 0, 0.003])  # adjust limits as needed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_effcai_euler(cai_trace, t, tau_effca=200.0, min_ca=70e-6, effcai0=0.0):
    """Euler method matching NEURON's METHOD euler."""
    n_points = len(t)
    effcai = np.zeros(n_points)
    effcai[0] = effcai0
    
    for i in range(n_points - 1):
        dt = t[i + 1] - t[i]
        driving = cai_trace[i] - min_ca
        deriv = -effcai[i] / tau_effca + driving
        effcai[i + 1] = effcai[i] + dt * deriv
    
    return effcai

# Check shapes first
print("cai_CR shape:", data["cai_CR"].shape)
print("effcai_GB shape:", data["effcai_GB"].shape)
print("t shape:", data["t"].shape)

# Get time and traces - handle shape correctly
t = data["t"]
cai = np.asarray(data["cai_CR"])
effcai_sim = np.asarray(data["effcai_GB"])

# If shape is (n_timepoints, n_synapses), transpose to (n_synapses, n_timepoints)
if cai.shape[0] == len(t):
    cai = cai.T
if effcai_sim.shape[0] == len(t):
    effcai_sim = effcai_sim.T

print("After transpose - cai shape:", cai.shape, "effcai_sim shape:", effcai_sim.shape)

# Now get first synapse
cai_trace = cai[0]
print("cai_trace length:", len(cai_trace), "t length:", len(t))

# Compute analytical solution
effcai_analytical = compute_effcai_euler(cai_trace, t, tau_effca=200.0, min_ca=70e-6)

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t/1000, cai_trace * 1e6, 'b-', linewidth=0.5)
axes[0].set_ylabel("cai_CR (nM)")
axes[0].set_title("Calcium Concentration")

axes[1].plot(t/1000, effcai_sim[0], 'k-', label='NEURON', linewidth=1)
axes[1].plot(t/1000, effcai_analytical, 'r--', label='Analytical (Euler)', linewidth=1)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("effcai_GB (μM)")
axes[1].set_title("Effective Calcium Comparison")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print error statistics
diff = effcai_sim[0] - effcai_analytical
print(f"\nMax absolute error: {np.max(np.abs(diff)):.6e}")
print(f"Mean absolute error: {np.mean(np.abs(diff)):.6e}")

In [ ]:
print("NaN in cai:", np.any(np.isnan(cai_trace)))
print("Inf in cai:", np.any(np.isinf(cai_trace)))
print("cai min/max:", np.min(cai_trace), np.max(cai_trace))
print("t min/max:", t.min(), t.max())
print("dt:", np.diff(t)[:5])

In [ ]:
# Zoom to first 50 seconds (50000 ms)
mask = t < 50000

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t[mask]/1000, cai_trace[mask] * 1e6, 'b-', linewidth=0.5)
axes[0].set_ylabel("cai_CR (nM)")
axes[0].set_title("Calcium Concentration")

axes[1].plot(t[mask]/1000, effcai_sim[0][mask], 'k-', label='NEURON', linewidth=1)
axes[1].plot(t[mask]/1000, effcai_analytical[mask], 'r--', label='Analytical (Euler)', linewidth=1)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("effcai_GB (μM)")
axes[1].set_title("Effective Calcium Comparison (First 50s)")
axes[1].legend()

plt.tight_layout()
plt.show()

# Error stats on this region
diff = effcai_sim[0][mask] - effcai_analytical[mask]
print(f"Max absolute error (first 50s): {np.max(np.abs(diff)):.6e}")